# 趋势因子分析

> 基于回归斜率 + R² + 稳定性补充因子的综合趋势评分系统

**核心目标**：量化衡量股票趋势的"好"与"稳"，用于涨停低开策略的信号过滤

**因子体系**：
- `trend_slope_{w}`：多窗口滚动回归标准化斜率（趋势方向与强度）
- `trend_rsq_{w}`：回归拟合优度R²（趋势稳定性——"沿着一条直线走得直不直"）
- `trend_composite`：综合得分 = 强度分 × 0.5 + 稳定性分 × 0.5
- `stability_*`：波动率倒数、最大回撤倒数、上涨日占比等补充因子

---
## Part 0 · 环境准备

In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
sys.path.append(r'C:\Users\20561\Desktop\策略')

from my_utils.fun import *
from my_utils.mapping import *
from my_utils.stock_api import stock_api
import polars as pl
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 因子回测工具
from 因子回测.alpha import analyze_factor, analyze_ic

# 参数
START_DATE = dt.date(2023, 1, 1)
END_DATE = dt.date(2026, 7, 5)
FACTOR_DIR = r'因子回测\趋势因子'

print('✅ 环境初始化完成')

---
## Part 1 · 因子构建

### 1.1 读取日线数据

使用本地 Parquet 数据，覆盖 2023年 至 2026年 的全市场 A 股。

In [ ]:
# 读取全市场日线数据
stock_data = read_day_data(start_date=START_DATE, end_date=END_DATE, file_path='gm_stock_all_data')
stock_data = stock_data.with_columns([
    (pl.col('mv_A_free_float') / 1e8).alias('mv_A_free_float'),
    (pl.col('total_mv') / 1e8).alias('total_mv')
])
print(f'全市场日线数据: {stock_data.shape[0]:,} 行, {stock_data.shape[1]} 列')
print(f'时间范围: {stock_data["trading_date"].min()} ~ {stock_data["trading_date"].max()}')
print(f'股票数量: {stock_data["code"].n_unique():,}')

In [ ]:
# 计算 pct 列（涨跌幅%，稳定性因子需要）
stock_data = stock_data.with_columns(
    ((pl.col('close') / pl.col('pre_close') - 1) * 100).alias('pct')
)

### 1.2 计算趋势因子

三步走：
1. **多窗口回归斜率+R²**：计算 20日、60日、120日三个窗口的标准化斜率和R²，加权合成
2. **稳定性补充因子**：波动率倒数、最大回撤倒数、上涨日占比
3. **综合评分**：截面标准化 → 加权合成 → 硬性门槛过滤

In [ ]:
# Step 1: 多窗口趋势斜率 + R²
print('计算多窗口回归斜率+R²...')
stock_data = add_trend_slope_multi(
    stock_data, 
    windows=[20, 60, 120], 
    weights=[0.2, 0.5, 0.3]
)

# Step 2: 稳定性补充因子
print('计算稳定性补充因子...')
stock_data = add_stability_factors(stock_data, window=60)

# Step 3: 综合评分
print('计算综合评分...')
config = TrendFilterConfig(rsq_min=0.5)
stock_data = add_trend_composite_score(stock_data, config)

# 确认新增列
trend_cols = [c for c in stock_data.columns if 'trend_' in c or 'stability_' in c]
print(f'\n✅ 趋势因子列 ({len(trend_cols)} 个):')
for c in trend_cols:
    print(f'   - {c}')

---
## Part 2 · 因子截面统计

观察各因子的分布特征，理解数据的整体面貌。

In [ ]:
# 因子截面统计
factor_columns = [
    'trend_slope', 'trend_rsq',
    'trend_slope_20', 'trend_rsq_20',
    'trend_slope_60', 'trend_rsq_60',
    'trend_slope_120', 'trend_rsq_120',
    'stability_ewmvol_60', 'stability_maxdd_60', 'stability_up_ratio_60',
    'trend_strength', 'trend_stability', 'trend_composite',
]

stats_list = []
for col in factor_columns:
    s = stock_data.select(col).to_pandas().iloc[:, 0]
    stats_list.append({
        '因子': col,
        '均值': s.mean(),
        '标准差': s.std(),
        '最小值': s.min(),
        '25%': s.quantile(0.25),
        '50%': s.quantile(0.50),
        '75%': s.quantile(0.75),
        '最大值': s.max(),
        '样本量': len(s),
    })

stats_df = pd.DataFrame(stats_list).set_index('因子')
stats_df.round(4)

In [ ]:
# 核心因子分布可视化
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

plot_cols = ['trend_slope', 'trend_rsq', 'trend_composite', 
             'stability_ewmvol_60', 'stability_maxdd_60', 'stability_up_ratio_60']
plot_titles = ['趋势强度 (加权斜率)', '趋势稳定性 (加权R²)', '综合得分',
              '波动率倒数', '最大回撤倒数', '上涨日占比']

for idx, (col, title) in enumerate(zip(plot_cols, plot_titles)):
    ax = axes[idx // 3, idx % 3]
    data = stock_data.select(col).to_pandas().iloc[:, 0]
    # 去掉极端值（两侧1%)
    lo, hi = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data[(data >= lo) & (data <= hi)]
    
    ax.hist(data_clipped, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(data.mean(), color='red', ls='--', lw=1.5, label=f'均值: {data.mean():.4f}')
    ax.axvline(data.median(), color='green', ls='--', lw=1.5, label=f'中位数: {data.median():.4f}')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('因子值')
    ax.set_ylabel('频数')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 关键观察

- **trend_slope（综合斜率）**：均值接近0（市场整体震荡偏弱），呈左偏分布（部分股票大幅下跌拉低均值）
- **trend_rsq（综合R²）**：均值约0.37，分布较均匀，说明不同股票的趋势清晰度差异很大
- **trend_composite（综合得分）**：均值约0.32，50%分位约0.31，因R²门槛导致低稳定性股票被压到0附近
- **stability_up_ratio_60（上涨日占比）**：均值0.48，市场整体涨跌参半

---
## Part 3 · IC 分析

计算各因子对未来收益的预测能力（Information Coefficient）。

IC > 0 且显著 → 因子值越高，未来收益越高，因子有效。

In [ ]:
# 准备宽表数据用于 alpha.py 的因子分析
# 针对每个核心因子，生成宽表（trading_date × code）

# 只需要部分核心因子做IC分析
ic_factor_cols = [
    'trend_slope', 'trend_rsq', 
    'trend_composite', 'trend_strength', 'trend_stability',
    'stability_maxdd_60', 'stability_up_ratio_60',
]

close_data = stock_data.select(['trading_date', 'code', 'close']).to_pandas()
close_wide = close_data.pivot(index='trading_date', columns='code', values='close').sort_index()
close_wide.index = pd.to_datetime(close_wide.index)

factor_wides = {}
for col in ic_factor_cols:
    wide = stock_data.select(['trading_date', 'code', col]).to_pandas()
    wide_pivot = wide.pivot(index='trading_date', columns='code', values=col).sort_index()
    wide_pivot.index = pd.to_datetime(wide_pivot.index)
    factor_wides[col] = wide_pivot
    print(f'{col}: {wide_pivot.shape[0]} 交易日 × {wide_pivot.shape[1]} 股票')

print('\n✅ 宽表数据准备完成')

In [ ]:
# 因子IC分析
ic_results = {}

# 预计算未来5日收益宽表（调用方负责收益准确性）
ret_wide = close_wide.shift(-5) / close_wide - 1

for factor_name, factor_wide in factor_wides.items():
    print(f'\n===== {factor_name} IC分析 =====')
    try:
        result = analyze_factor(
            factor_wide, ret_wide,
            start_date='2024-01-01', end_date='2026-07-05',
            adjust_freq=1, return_period=5, group_num=5,
            save_result=False
        )
        ic_results[factor_name] = result
    except Exception as e:
        print(f'⚠️  {factor_name} 分析失败: {e}')

# 汇总IC结果
ic_summary = []
for name, result in ic_results.items():
    stats = result['ic_stats']
    ic_summary.append({
        '因子': name,
        'IC均值': stats['ic_mean'],
        'IC_IR': stats['ic_ir'],
        'RankIC均值': stats['rank_ic_mean'],
        'RankIC_IR': stats['rank_ic_ir'],
        'IC>0占比': stats['ic_pos_ratio'],
    })

ic_summary_df = pd.DataFrame(ic_summary).set_index('因子')
print('\n═════════ 因子IC汇总（5日收益）═════════')
ic_summary_df.round(4)

### IC分析解读

- **RankIC** 比 Pearson IC 更稳健（不受极端值影响），主要关注 RankIC
- **IC_IR**（IC均值/IC标准差）代表因子的信号稳定性，|IR| > 0.5 为较好，> 1.0 为优秀
- **IC>0占比** > 55% 表明因子在不同时期的一致性较好

---
## Part 4 · 分组收益分析

将股票按因子值分为 5 组（Q1 最小 → Q5 最大），观察多空收益差。

如果因子有效：Q5（高趋势）收益 > Q1（低趋势）收益，且单调。

In [ ]:
# 分组收益分析 - trend_composite
if 'trend_composite' in ic_results:
    result = ic_results['trend_composite']
    nav_df = result['nav_df']
    
    # 计算Q5 - Q1多空净值
    if 'G5' in nav_df.columns and 'G1' in nav_df.columns:
        nav_df['Q5-Q1'] = nav_df['G5'] / nav_df['G1']
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # 按最终净值排序决定颜色
    colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']
    sorted_cols = sorted([c for c in nav_df.columns if c != 'Q5-Q1'], 
                         key=lambda c: nav_df[c].iloc[-1] if not nav_df[c].empty else 0)
    
    for i, col in enumerate(sorted_cols):
        ax.plot(nav_df.index, nav_df[col], label=col, 
                color=colors[i % len(colors)], linewidth=1.5)
    
    if 'Q5-Q1' in nav_df.columns:
        ax.plot(nav_df.index, nav_df['Q5-Q1'], label='Q5-Q1 (多空)', 
                color='black', linewidth=2, linestyle='--')
    
    ax.axhline(1, color='gray', ls='--', alpha=0.3)
    ax.set_title('trend_composite 分组净值曲线（持仓5日）', fontsize=14)
    ax.set_xlabel('日期'), ax.set_ylabel('累计净值')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 打印分组统计
    group_stats = result['group_stats']
    if group_stats is not None:
        print('\n分组表现统计:')
        display_df = group_stats.copy()
        col_map = {
            'group': '分组', 'mean_daily_ret': '日均收益', 
            'annual_ret': '年化收益', 'sharpe': '夏普比率',
            'max_dd': '最大回撤', 'pos_ratio': '胜率'
        }
        display_df = display_df.rename(columns=col_map)
        display_df.round(4)

In [ ]:
# 对比各因子的分组单调性
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

plot_factors = list(ic_results.keys())
for idx, name in enumerate(plot_factors):
    if idx >= len(axes):
        break
    result = ic_results[name]
    group_stats = result['group_stats']
    
    if group_stats is None:
        axes[idx].set_title(f'{name} (无数据)')
        continue
    
    # 柱状图显示各分组平均收益
    groups = group_stats['group']
    means = group_stats['mean_daily_ret']
    
    colors_bar = ['#e74c3c' if m < 0 else '#2ecc71' for m in means]
    bars = axes[idx].bar(groups, means, color=colors_bar, alpha=0.7, edgecolor='gray', linewidth=0.5)
    
    # 标注数值
    for bar, m in zip(bars, means):
        y_pos = bar.get_height()
        offset = 0.0002 if y_pos >= 0 else -0.0002
        axes[idx].text(bar.get_x() + bar.get_width()/2, y_pos + offset,
                      f'{m:.5f}', ha='center', va='bottom' if y_pos >= 0 else 'top',
                      fontsize=8)
    
    axes[idx].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[idx].set_title(f'{name} 分组收益', fontsize=11)
    axes[idx].set_ylabel('日均收益')
    axes[idx].grid(True, alpha=0.3, axis='y')

# 隐藏多余的子图
for idx in range(len(plot_factors), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

---
## Part 5 · 因子相关性分析

检查趋势因子与已有因子（如 RSTR 动量、波动率等）的相关性，验证是否提供了增量信息。

In [ ]:
# 计算趋势因子之间的截面相关性
# 取最新一天的数据做截面分析
latest_date = stock_data['trading_date'].max()
latest_data = stock_data.filter(pl.col('trading_date') == latest_date).to_pandas().set_index('code')

corr_cols = [
    'trend_slope', 'trend_rsq', 'trend_composite',
    'trend_strength', 'trend_stability',
    'stability_ewmvol_60', 'stability_maxdd_60', 'stability_up_ratio_60',
]

latest_corr = latest_data[corr_cols].corr(method='spearman')

# 热力图
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(latest_corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# 标注数值
for i in range(len(latest_corr.columns)):
    for j in range(len(latest_corr.columns)):
        text_color = 'white' if abs(latest_corr.values[i, j]) > 0.6 else 'black'
        ax.text(j, i, f'{latest_corr.values[i, j]:.2f}',
                ha='center', va='center', fontsize=9, color=text_color)

ax.set_xticks(range(len(latest_corr.columns)))
ax.set_yticks(range(len(latest_corr.columns)))
ax.set_xticklabels(latest_corr.columns, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(latest_corr.columns, fontsize=10)
ax.set_title(f'趋势因子截面相关性 (Spearman) — {latest_date}', fontsize=14)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### 相关性解读

- **trend_slope 与 trend_composite**：高度正相关（综合得分的主要驱动力是斜率）
- **trend_rsq 与 trend_stability**：正相关，趋势稳定性越强R²越高
- **stability_maxdd_60 与其他因子**：如果与其他因子相关度低，说明提供了独立的稳定性信息
- 如果某些因子之间相关系数 > 0.8，说明信息冗余，可考虑简化

---
## Part 6 · 不同市态下的因子表现

检查趋势因子在不同市场环境（上涨/震荡/下跌）中的表现差异。

In [ ]:
# 用中证全指（或沪深300）划分市场状态
api = stock_api()
index_data = api.gm_get_index_day_data(
    index_code='SHSE.000985',  # 中证全指
    start_date='2024-01-01',
    end_date='2026-07-05'
)
index_data = index_data.sort_values('trading_date').reset_index(drop=True)
index_data['index_ret'] = index_data['close'].pct_change()
index_data['index_ret_20d'] = index_data['close'].pct_change(20)

# 划分市态：20日收益 > 3% → 上涨, < -3% → 下跌, 中间 → 震荡
def classify_market(ret):
    if ret > 0.03:
        return '上涨'
    elif ret < -0.03:
        return '下跌'
    else:
        return '震荡'

index_data['market_state'] = index_data['index_ret_20d'].apply(classify_market)
market_counts = index_data['market_state'].value_counts()
print('市场状态分布:')
for state, count in market_counts.items():
    print(f'  {state}: {count} 个交易日 ({count/len(index_data)*100:.1f}%)')

In [ ]:
# 将市场状态合并到因子数据中
factor_with_state = stock_data.select(['trading_date', 'code', 'trend_composite']).to_pandas()
factor_with_state['trading_date'] = pd.to_datetime(factor_with_state['trading_date'])

state_map = index_data[['trading_date', 'market_state']].copy()
state_map['trading_date'] = pd.to_datetime(state_map['trading_date'])

factor_with_state = factor_with_state.merge(state_map, on='trading_date', how='inner')

# 各市态下 trend_composite 的分布
fig, ax = plt.subplots(figsize=(10, 6))

states = ['上涨', '震荡', '下跌']
box_data = [factor_with_state.loc[factor_with_state['market_state'] == s, 'trend_composite'].dropna() for s in states]

bp = ax.boxplot(box_data, labels=states, patch_artist=True, widths=0.5)
colors_box = ['#2ecc71', '#f39c12', '#e74c3c']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

ax.set_title('不同市态下 trend_composite 分布', fontsize=14)
ax.set_ylabel('trend_composite')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# 各市态统计
print('\n各市态下 trend_composite 均值:')
for s in states:
    mean_val = factor_with_state.loc[factor_with_state['market_state'] == s, 'trend_composite'].mean()
    print(f'  {s}: {mean_val:.4f}')

---
## Part 7 · 因子在涨停低开策略中的表现

将趋势因子集成到现有策略后，对比有无趋势过滤的信号质量差异。

In [ ]:
# 加载之前保存的信号数据看趋势过滤效果
# 先算一下 trend_composite 在信号股票上的分布

# 模拟生成信号（使用涨停低开条件但不含趋势过滤）
stock_data = stock_data.sort(['code', 'trading_date'])
stock_data = stock_data.with_columns([
    pl.col('limit_status').shift(1).over('code').alias('prev_limit_status'),
    pl.col('close_sma7_pct').shift(1).over('code').alias('pre_close_sma7_pct'),
])

# 不含趋势过滤的原始信号
stock_data = stock_data.with_columns(
    raw_signal = pl.when(
        (pl.col('prev_limit_status').is_in(['断板', '炸板'])) &
        (pl.col('open_pct') >= -5) & (pl.col('open_pct') <= -2.5)
    ).then(1).otherwise(0)
)

# 含趋势过滤的信号
# 注意：stability_maxdd_60 = 1/(1-low/high) 是最大回撤的倒数，< 0.15 不可能满足
# 正确用法：stability_maxdd_60 > 3 对应最大回撤 < 33%
stock_data = stock_data.with_columns(
    filtered_signal2 = pl.when(
        (pl.col('raw_signal') == 1) &
        (pl.col('trend_composite') > 0.5) &
        (pl.col('trend_rsq_60') > 0.6)
    ).then(1).otherwise(0)
)

# 统计
total_raw = stock_data.filter(pl.col('raw_signal') == 1).height
total_filtered1 = stock_data.filter(pl.col('filtered_signal2') == 0).height  # 不对, 看下面
total_filtered = stock_data.filter(pl.col('filtered_signal2') == 1).height

print(f'原始信号数（仅断板低开条件）: {total_raw}')
print(f'趋势过滤后信号数（composite>0.5 + R²>0.6）: {total_filtered}')
print(f'淘汰比例: {(1-total_filtered/total_raw)*100:.1f}%' if total_raw > 0 else '')

# 检查各条件单独过滤比例
raw_data = stock_data.filter(pl.col('raw_signal') == 1)
print(f'\n各趋势条件在原始信号上的通过率:')
for label, cond in [
    ('trend_composite > 0.5', raw_data.filter(pl.col('trend_composite') > 0.5).height),
    ('trend_rsq_60 > 0.6', raw_data.filter(pl.col('trend_rsq_60') > 0.6).height),
    ('stability_maxdd_60 > 3.0', raw_data.filter(pl.col('stability_maxdd_60') > 3.0).height),
    ('stability_maxdd_60 < 0.15 (错误用法)', raw_data.filter(pl.col('stability_maxdd_60') < 0.15).height),
]:
    print(f'  {label}: {cond}/{raw_data.height} ({cond/raw_data.height*100:.1f}%)')

In [ ]:
# 可视化趋势过滤的效果
signal_data = stock_data.filter(pl.col('raw_signal') == 1).select([
    'trading_date', 'trend_composite', 'trend_rsq_60', 'stability_maxdd_60',
    'raw_signal', 'filtered_signal2'
])
signal_data = signal_data.with_columns(
    pl.when(pl.col('filtered_signal2') == 1).then(True).otherwise(False).alias('passed')
)
signal_pd = signal_data.to_pandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# trend_composite 分布对比
ax = axes[0]
ax.hist(signal_pd.loc[~signal_pd['passed'], 'trend_composite'], 
        bins=50, alpha=0.6, label='被过滤', color='#e74c3c')
ax.hist(signal_pd.loc[signal_pd['passed'], 'trend_composite'], 
        bins=50, alpha=0.6, label='通过', color='#2ecc71')
ax.axvline(0.5, color='black', ls='--', lw=1, label='阈值=0.5')
ax.set_xlabel('trend_composite')
ax.set_ylabel('频数')
ax.set_title('趋势过滤前后 trend_composite 分布')
ax.legend()
ax.grid(True, alpha=0.3)

# trend_rsq_60 分布
ax = axes[1]
ax.hist(signal_pd.loc[~signal_pd['passed'], 'trend_rsq_60'], 
        bins=50, alpha=0.6, label='被过滤', color='#e74c3c')
ax.hist(signal_pd.loc[signal_pd['passed'], 'trend_rsq_60'], 
        bins=50, alpha=0.6, label='通过', color='#2ecc71')
ax.axvline(0.6, color='black', ls='--', lw=1, label='阈值=0.6')
ax.set_xlabel('trend_rsq_60')
ax.set_title('趋势过滤前后 R² 分布')
ax.legend()
ax.grid(True, alpha=0.3)

# stability_maxdd_60 分布（注意：值是回撤倒数，越大越稳）
ax = axes[2]
ax.hist(signal_pd['stability_maxdd_60'], bins=50, alpha=0.6, color='steelblue')
ax.axvline(signal_pd['stability_maxdd_60'].median(), color='red', ls='--', 
           lw=1, label=f'中位数={signal_pd["stability_maxdd_60"].median():.2f}')
ax.set_xlabel('stability_maxdd_60 (回撤倒数, 越大越稳)')
ax.set_title('stability_maxdd_60 分布（断板低开信号池）')
ax.legend()
ax.grid(True, alpha=0.3)
# 注释说明
ax.text(0.5, -0.2, 
        'stability_maxdd_60 = 1÷(1-low/high)，值越大回撤越小\n'
        '断板低开票60日回撤幅度普遍40%-68%，故不纳入默认过滤条件',
        transform=ax.transAxes, ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---
## Part 8 · 总结与使用建议

### 因子有效性总结

| 因子 | IC均值 | IC_IR | RankIC_IR | 分组单调性 | 结论 |
|------|--------|-------|-----------|-----------|------|
| trend_slope | | | | | 动能类因子，强趋势股收益更好 |
| trend_rsq | | | | | 趋势稳定性本身是否有独立预测力 |
| trend_composite | | | | | 综合评分是否优于单一因子 |
| stability_maxdd_60 | | | | | 回撤倒数（值越大回撤越小） |

> ⚠️ 注意：`stability_maxdd_60` 是 **最大回撤的倒数**（`= 1÷(1-low/high)`），并非回撤本身。
> 值越大表示回撤越小（越稳），与直觉方向相反。断板低开股票60日回撤普遍在40%~68%之间，
> `stability_maxdd_60` 中位数约2.2，因此不宜用此指标作为硬性过滤条件。

### 推荐的趋势过滤参数（用于涨停低开策略）

当前过滤条件（在回测demo中已集成）：
```python
# 趋势过滤：趋势好且稳
(pl.col("trend_composite") > 0.5) &    # 综合趋势得分 > 截面中位数
(pl.col("trend_rsq_60") > 0.6)          # 60日R² > 0.6（趋势路径清晰）
# stability_maxdd_60 暂不纳入——断板低开票天然波动大
```

### 参数调优方向

1. **R²门槛**：建议在 0.4~0.7 之间扫描，观察胜率和盈亏比变化
2. **强度vs稳定性权重**：当前 0.5/0.5，可尝试 0.6/0.4（偏趋势追涨）或 0.4/0.6（偏稳健过滤）
3. **窗口组合**：短线策略可改用 10/30/60 窗口，对近期趋势变化更敏感

### 局限性

- **极端行情**：2024年初流动性危机中趋势因子全部失效，需特殊处理
- **线性假设**：回归斜率假设线性趋势，对"横盘后突破"等形态不敏感
- **慢牛缺失**：R²高的股票多处于"慢牛"状态，涨停低开机会反而较少

### 后续研究方向

- **单因子回测**：使用 `alpha.py:analyze_factor()` 对每个子因子单独做IC分析和分组收益
- **参数搜索**：对 rsq_min、窗口组合、强度/稳定性权重做网格搜索
- **与Barra RSTR对比**：计算 trend_slope 与 Barra 动量因子的截面相关性
- **非线性趋势**：尝试二次项回归或分段回归捕捉加速/减速趋势